<a href="https://colab.research.google.com/github/tanveerratul34/Practice-/blob/main/Traffic_Conflict_Analysis_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# CELL 1: Setup — mount Drive, install packages, set paths
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pandas numpy

import os, re, glob
import numpy as np
import pandas as pd

BASE_DIR = "/content/drive/My Drive/Colab Notebooks/Safety Vissim"
OUTPUT_DIR = f"{BASE_DIR}/Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GEOM_PATH = f"{BASE_DIR}/conflict_area_geometry_with_angle.csv"
FZP_GLOB = f"{BASE_DIR}/*.fzp"

fzp_files = sorted(glob.glob(FZP_GLOB))
print(f"Found {len(fzp_files)} fzp files")
print(f"Geometry file exists: {os.path.exists(GEOM_PATH)}")
for f in fzp_files:
    print(" -", os.path.basename(f))

Mounted at /content/drive
Found 30 fzp files
Geometry file exists: True
 - Thesis vissim 3.0 (5)_001.fzp
 - Thesis vissim 3.0 (5)_002.fzp
 - Thesis vissim 3.0 (5)_003.fzp
 - Thesis vissim 3.0 (5)_004.fzp
 - Thesis vissim 3.0 (5)_005.fzp
 - Thesis vissim 3.0 (5)_006.fzp
 - Thesis vissim 3.0 (5)_007.fzp
 - Thesis vissim 3.0 (5)_008.fzp
 - Thesis vissim 3.0 (5)_009.fzp
 - Thesis vissim 3.0 (5)_010.fzp
 - Thesis vissim 3.0 (5)_011.fzp
 - Thesis vissim 3.0 (5)_012.fzp
 - Thesis vissim 3.0 (5)_013.fzp
 - Thesis vissim 3.0 (5)_014.fzp
 - Thesis vissim 3.0 (5)_015.fzp
 - Thesis vissim 3.0 (5)_016.fzp
 - Thesis vissim 3.0 (5)_017.fzp
 - Thesis vissim 3.0 (5)_018.fzp
 - Thesis vissim 3.0 (5)_019.fzp
 - Thesis vissim 3.0 (5)_020.fzp
 - Thesis vissim 3.0 (5)_021.fzp
 - Thesis vissim 3.0 (5)_022.fzp
 - Thesis vissim 3.0 (5)_023.fzp
 - Thesis vissim 3.0 (5)_024.fzp
 - Thesis vissim 3.0 (5)_025.fzp
 - Thesis vissim 3.0 (5)_026.fzp
 - Thesis vissim 3.0 (5)_027.fzp
 - Thesis vissim 3.0 (5)_028.fzp
 - T

In [ ]:
CHUNK_SIZE = 500_000

def run_id_from_filename(path):
    m = re.search(r"_(\d{3})\.fzp$", path)
    return int(m.group(1)) if m else None

def build_link_station_lookup(geom_df):
    lookup = {}
    for _, row in geom_df.iterrows():
        lookup.setdefault(int(row.LinkA), []).append((int(row.ConflictArea), row.StationA))
        lookup.setdefault(int(row.LinkB), []).append((int(row.ConflictArea), row.StationB))
    return lookup

def process_one_file(fzp_path, run_id, link_lookup, relevant_links):
    print(f"\n--- Processing RunID {run_id}: {os.path.basename(fzp_path)} ---")
    with open(fzp_path, encoding="utf-8-sig") as f:
        header_line_idx = None
        for i, line in enumerate(f):
            if line.startswith("$VEHICLE:"):
                header_line_idx = i
                break
    header = ["SIMSEC", "NO", "LINK", "LANE_INDEX", "POS", "POSLAT",
              "ACCEL", "CFX", "CFY", "CFZ", "CRX", "CRY", "CRZ", "SPEED"]

    crossing_records = []
    last_seen = {}

    reader = pd.read_csv(
        fzp_path, sep=";", skiprows=header_line_idx + 1, names=header,
        encoding="utf-8-sig", chunksize=CHUNK_SIZE, engine="c",
    )

    total_rows = 0
    for chunk_no, chunk in enumerate(reader):
        total_rows += len(chunk)
        chunk = chunk[chunk["LINK"].isin(relevant_links)]
        if chunk.empty:
            continue
        chunk = chunk.sort_values(["NO", "LINK", "SIMSEC"])

        for (veh, link), grp in chunk.groupby(["NO", "LINK"], sort=False):
            grp = grp.reset_index(drop=True)
            key = (veh, link)
            prev = last_seen.get(key)
            rows = []
            if prev is not None:
                rows.append(prev)
            rows.extend(grp.to_dict("records"))

            stations = link_lookup[link]
            for i in range(len(rows) - 1):
                r0, r1 = rows[i], rows[i + 1]
                pos0, pos1 = r0["POS"], r1["POS"]
                t0, t1 = r0["SIMSEC"], r1["SIMSEC"]
                sp0, sp1 = r0["SPEED"], r1["SPEED"]
                if pos1 == pos0:
                    continue
                for ca_no, station in stations:
                    if (pos0 - station) * (pos1 - station) <= 0:
                        frac = (station - pos0) / (pos1 - pos0)
                        cross_time = t0 + frac * (t1 - t0)
                        cross_speed = sp0 + frac * (sp1 - sp0)
                        crossing_records.append({
                            "RunID": run_id, "ConflictArea": ca_no, "Link": link,
                            "Veh": veh, "CrossTime": round(cross_time, 3),
                            "CrossSpeed_kmh": round(cross_speed, 2),
                        })
            last_seen[key] = rows[-1]

        if chunk_no % 3 == 0:
            print(f"  ...{total_rows:,} rows scanned so far, {len(crossing_records):,} crossings found")

    df = pd.DataFrame(crossing_records).drop_duplicates(subset=["RunID", "ConflictArea", "Link", "Veh"])
    print(f"  Run {run_id} complete: {len(df):,} crossing events")
    return df


geom_df = pd.read_csv(GEOM_PATH)
link_lookup = build_link_station_lookup(geom_df)
relevant_links = set(link_lookup.keys())
print(f"{len(relevant_links)} distinct links/connectors involved in conflict areas")

all_crossing_dfs = []
for fzp_path in fzp_files:
    run_id = run_id_from_filename(fzp_path)
    if run_id is None:
        print(f"WARNING: could not parse RunID from {fzp_path}, skipping")
        continue
    df = process_one_file(fzp_path, run_id, link_lookup, relevant_links)
    all_crossing_dfs.append(df)

vehicle_crossings = pd.concat(all_crossing_dfs, ignore_index=True)
vehicle_crossings.to_csv(f"{OUTPUT_DIR}/vehicle_crossing_times.csv", index=False)

print(f"\n=== Combined across {len(fzp_files)} runs ===")
print(f"Total crossing events: {len(vehicle_crossings):,}")
print(vehicle_crossings.groupby("RunID").size())
print(f"\nSaved to {OUTPUT_DIR}/vehicle_crossing_times.csv")

33 distinct links/connectors involved in conflict areas

--- Processing RunID 1: Thesis vissim 3.0 (5)_001.fzp ---
  ...500,000 rows scanned so far, 7,576 crossings found
  ...1,676,910 rows scanned so far, 20,305 crossings found
  Run 1 complete: 19,469 crossing events

--- Processing RunID 2: Thesis vissim 3.0 (5)_002.fzp ---
  ...500,000 rows scanned so far, 7,485 crossings found
  ...1,942,351 rows scanned so far, 15,375 crossings found
  Run 2 complete: 14,834 crossing events

--- Processing RunID 3: Thesis vissim 3.0 (5)_003.fzp ---
  ...500,000 rows scanned so far, 7,285 crossings found
  ...1,785,464 rows scanned so far, 18,141 crossings found
  Run 3 complete: 17,375 crossing events

--- Processing RunID 4: Thesis vissim 3.0 (5)_004.fzp ---
  ...500,000 rows scanned so far, 7,383 crossings found
  ...1,721,140 rows scanned so far, 17,769 crossings found
  Run 4 complete: 17,063 crossing events

--- Processing RunID 5: Thesis vissim 3.0 (5)_005.fzp ---
  ...500,000 rows scanned

In [ ]:
TTC_PET_THRESHOLD = 1.5  # seconds; SSAM-style conflict cutoff (Gettman & Head, 2003)

ca_links = geom_df.set_index("ConflictArea")[["LinkA", "LinkB"]].to_dict("index")

records = []
skipped_same_link = 0

for (run_id, ca), grp in vehicle_crossings.groupby(["RunID", "ConflictArea"]):
    if ca not in ca_links:
        continue
    link_a_id = ca_links[ca]["LinkA"]
    link_b_id = ca_links[ca]["LinkB"]

    approach_a = grp[grp.Link == link_a_id].sort_values("CrossTime").reset_index(drop=True)
    approach_b = grp[grp.Link == link_b_id].sort_values("CrossTime").reset_index(drop=True)
    if approach_a.empty or approach_b.empty:
        continue

    combined = pd.concat([approach_a.assign(Approach="A"), approach_b.assign(Approach="B")])
    combined = combined.sort_values("CrossTime").reset_index(drop=True)

    n = len(combined)
    for i in range(n - 1):
        veh_a = combined.loc[i]
        veh_b = combined.loc[i + 1]
        if veh_a.Approach == veh_b.Approach:
            skipped_same_link += 1
            continue
        gap = veh_b.CrossTime - veh_a.CrossTime

        speed_a_ms = veh_a.CrossSpeed_kmh / 3.6
        speed_b_ms = veh_b.CrossSpeed_kmh / 3.6
        rel_speed = abs(speed_a_ms - speed_b_ms)

        if gap <= TTC_PET_THRESHOLD:
            gap_distance = gap * speed_a_ms
            ttc = gap_distance / rel_speed if rel_speed > 1e-3 else gap
            pet = np.nan
        else:
            pet = gap
            ttc = np.nan

        conflict = int(
            (not np.isnan(ttc) and ttc < TTC_PET_THRESHOLD) or
            (not np.isnan(pet) and pet < TTC_PET_THRESHOLD)
        )

        records.append({
            "RunID": run_id, "ConflictArea": ca, "LinkA": veh_a.Link, "LinkB": veh_b.Link,
            "VehA": int(veh_a.Veh), "VehB": int(veh_b.Veh),
            "CrossTimeA": veh_a.CrossTime, "CrossTimeB": veh_b.CrossTime,
            "TimeGap": round(gap, 3), "SpeedA_kmh": veh_a.CrossSpeed_kmh, "SpeedB_kmh": veh_b.CrossSpeed_kmh,
            "RelativeSpeed_ms": round(rel_speed, 3),
            "TTC": round(ttc, 3) if not np.isnan(ttc) else np.nan,
            "PET": round(pet, 3) if not np.isnan(pet) else np.nan,
            "Conflict": conflict,
        })

labeled_dataset = pd.DataFrame(records)

print(f"Skipped {skipped_same_link:,} same-approach pairs (car-following)")
print(f"\nTotal vehicle-pair interactions: {len(labeled_dataset):,}")
print(f"\nPer-run breakdown:")
print(labeled_dataset.groupby("RunID").agg(interactions=("Conflict", "size"), conflicts=("Conflict", "sum")))
print(f"\nOverall conflict rate: {labeled_dataset.Conflict.mean():.1%}")

Skipped 320,677 same-approach pairs (car-following)

Total vehicle-pair interactions: 106,095

Per-run breakdown:
       interactions  conflicts
RunID                         
1              3949        774
2              2906        504
3              3449        683
4              3442        556
5              3858        785
6              3179        718
7              3559        746
8              3494        669
9              3016        553
10             3565        616
11             3647        712
12             3757        651
13             3876        791
14             3754        699
15             3549        662
16             3391        649
17             3599        754
18             3796        745
19             3804        619
20             3620        740
21             3393        674
22             3863        718
23             3446        585
24             3575        693
25             3472        692
26             3492        669
27             376

In [ ]:
LOOKBACK_SECONDS = 8.0
VEHICLE_LENGTH_M = 4.5

def load_needed_trajectories(fzp_path, needed_keys):
    with open(fzp_path, encoding="utf-8-sig") as f:
        header_line_idx = None
        for i, line in enumerate(f):
            if line.startswith("$VEHICLE:"):
                header_line_idx = i
                break
    header = ["SIMSEC", "NO", "LINK", "LANE_INDEX", "POS", "POSLAT",
              "ACCEL", "CFX", "CFY", "CFZ", "CRX", "CRY", "CRZ", "SPEED"]
    needed_vehs = {v for v, l in needed_keys}
    needed_links = {l for v, l in needed_keys}
    collected = {k: [] for k in needed_keys}

    reader = pd.read_csv(
        fzp_path, sep=";", skiprows=header_line_idx + 1, names=header,
        encoding="utf-8-sig", chunksize=CHUNK_SIZE, engine="c",
        usecols=["SIMSEC", "NO", "LINK", "POS", "SPEED"],
    )
    for chunk in reader:
        chunk = chunk[chunk["NO"].isin(needed_vehs) & chunk["LINK"].isin(needed_links)]
        if chunk.empty:
            continue
        for (veh, link), grp in chunk.groupby(["NO", "LINK"]):
            key = (veh, link)
            if key in collected:
                collected[key].append(grp[["SIMSEC", "POS", "SPEED"]])

    result = {}
    for key, parts in collected.items():
        if parts:
            df = pd.concat(parts, ignore_index=True).sort_values("SIMSEC").drop_duplicates("SIMSEC")
            result[key] = df.reset_index(drop=True)
    return result

def compute_crossing_ttc(traj_1, station_1, traj_2, station_2, lookback_seconds):
    df1 = traj_1.copy(); df1["dist_to_conflict"] = station_1 - df1["POS"]
    df2 = traj_2.copy(); df2["dist_to_conflict"] = station_2 - df2["POS"]

    t_cross = max(df1["SIMSEC"].max(), df2["SIMSEC"].max())
    t_min = t_cross - lookback_seconds

    df1 = df1[(df1["SIMSEC"] >= t_min) & (df1["dist_to_conflict"] >= 0) & (df1["SPEED"] > 0.5)]
    df2 = df2[(df2["SIMSEC"] >= t_min) & (df2["dist_to_conflict"] >= 0) & (df2["SPEED"] > 0.5)]
    if df1.empty or df2.empty:
        return np.nan

    merged = pd.merge(df1, df2, on="SIMSEC", suffixes=("_1", "_2"))
    if merged.empty:
        return np.nan

    speed_1_ms = merged["SPEED_1"] / 3.6
    speed_2_ms = merged["SPEED_2"] / 3.6
    entry_time_1 = merged["dist_to_conflict_1"] / speed_1_ms
    entry_time_2 = merged["dist_to_conflict_2"] / speed_2_ms
    exit_time_1 = entry_time_1 + (VEHICLE_LENGTH_M / speed_1_ms)
    exit_time_2 = entry_time_2 + (VEHICLE_LENGTH_M / speed_2_ms)

    veh1_first = entry_time_1 < entry_time_2
    ttc_values = pd.Series(np.nan, index=merged.index)

    case_a = veh1_first & (exit_time_1 > entry_time_2)
    ttc_values[case_a] = entry_time_2[case_a]
    case_b = (~veh1_first) & (exit_time_2 > entry_time_1)
    ttc_values[case_b] = entry_time_1[case_b]

    ttc_values = ttc_values.dropna()
    return float(ttc_values.min()) if not ttc_values.empty else np.nan


geom_indexed = geom_df.set_index("ConflictArea")
candidates = labeled_dataset[labeled_dataset["TTC"].notna()].copy()
print(f"{len(candidates):,} candidate pairs need trajectory-based TTC")

labeled_dataset["TTC_method"] = np.where(labeled_dataset["TTC"].notna(), "gap_approx", "not_applicable")

run_files = {run_id_from_filename(f): f for f in fzp_files}
results = {}
not_defined_count = 0

for run_id, run_candidates in candidates.groupby("RunID"):
    if run_id not in run_files:
        continue
    fzp_path = run_files[run_id]
    print(f"\n--- RunID {run_id}: {len(run_candidates)} candidate pairs ---")

    needed_keys = set()
    for _, row in run_candidates.iterrows():
        needed_keys.add((row.VehA, row.LinkA))
        needed_keys.add((row.VehB, row.LinkB))

    trajectories = load_needed_trajectories(fzp_path, needed_keys)
    print(f"  Loaded {len(trajectories)} trajectory segments")

    for idx, row in run_candidates.iterrows():
        ca = row.ConflictArea
        if ca not in geom_indexed.index:
            continue
        station_a = geom_indexed.loc[ca, "StationA"] if geom_indexed.loc[ca, "LinkA"] == row.LinkA else geom_indexed.loc[ca, "StationB"]
        station_b = geom_indexed.loc[ca, "StationB"] if geom_indexed.loc[ca, "LinkB"] == row.LinkB else geom_indexed.loc[ca, "StationA"]

        key_a = (row.VehA, row.LinkA)
        key_b = (row.VehB, row.LinkB)
        if key_a not in trajectories or key_b not in trajectories:
            continue

        ttc_val = compute_crossing_ttc(trajectories[key_a], station_a, trajectories[key_b], station_b, LOOKBACK_SECONDS)
        if not np.isnan(ttc_val):
            results[idx] = ttc_val
        else:
            not_defined_count += 1

print(f"\nTrajectory-based TTC defined for {len(results):,} / {len(candidates):,} candidates")

for idx, val in results.items():
    labeled_dataset.loc[idx, "TTC"] = round(val, 3)
    labeled_dataset.loc[idx, "TTC_method"] = "trajectory"

labeled_dataset["Conflict"] = (
    ((labeled_dataset["TTC"].notna()) & (labeled_dataset["TTC"] < 1.5)) |
    ((labeled_dataset["PET"].notna()) & (labeled_dataset["PET"] < 1.5))
).astype(int)

print(f"\nTTC method breakdown:\n{labeled_dataset['TTC_method'].value_counts()}")
print(f"\nUpdated conflict rate: {labeled_dataset['Conflict'].mean():.1%}")

labeled_dataset.to_csv(f"{OUTPUT_DIR}/traffic_conflict_dataset_final.csv", index=False)
print(f"\nSaved to {OUTPUT_DIR}/traffic_conflict_dataset_final.csv")


59,427 candidate pairs need trajectory-based TTC

--- RunID 1: 2278 candidate pairs ---
  Loaded 2121 trajectory segments

--- RunID 2: 1525 candidate pairs ---
  Loaded 1441 trajectory segments

--- RunID 3: 1922 candidate pairs ---
  Loaded 1833 trajectory segments

--- RunID 4: 1808 candidate pairs ---
  Loaded 1844 trajectory segments

--- RunID 5: 2291 candidate pairs ---
  Loaded 2141 trajectory segments

--- RunID 6: 1815 candidate pairs ---
  Loaded 1703 trajectory segments

--- RunID 7: 2099 candidate pairs ---
  Loaded 2025 trajectory segments

--- RunID 8: 1921 candidate pairs ---
  Loaded 1827 trajectory segments

--- RunID 9: 1718 candidate pairs ---
  Loaded 1689 trajectory segments

--- RunID 10: 1846 candidate pairs ---
  Loaded 1804 trajectory segments

--- RunID 11: 2020 candidate pairs ---
  Loaded 1983 trajectory segments

--- RunID 12: 2069 candidate pairs ---
  Loaded 1973 trajectory segments

--- RunID 13: 2219 candidate pairs ---
  Loaded 2065 trajectory segment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
BASE_DIR = "/content/drive/My Drive/Colab Notebooks/Safety Vissim"
print("BASE_DIR exists:", os.path.exists(BASE_DIR))
if os.path.exists(BASE_DIR):
    print("Contents:")
    for f in os.listdir(BASE_DIR):
        print(" -", f)

BASE_DIR exists: True
Contents:
 - conflict_area_geometry_with_angle.csv
 - Thesis vissim 3.0 (5)_001.fzp
 - Thesis vissim 3.0 (5)_002.fzp
 - Thesis vissim 3.0 (5)_003.fzp
 - Thesis vissim 3.0 (5)_004.fzp
 - Thesis vissim 3.0 (5)_005.fzp
 - Thesis vissim 3.0 (5)_006.fzp
 - Thesis vissim 3.0 (5)_007.fzp
 - Thesis vissim 3.0 (5)_008.fzp
 - Thesis vissim 3.0 (5)_009.fzp
 - Thesis vissim 3.0 (5)_010.fzp
 - Thesis vissim 3.0 (5)_011.fzp
 - Thesis vissim 3.0 (5)_012.fzp
 - Thesis vissim 3.0 (5)_013.fzp
 - Thesis vissim 3.0 (5)_014.fzp
 - Thesis vissim 3.0 (5)_015.fzp
 - Thesis vissim 3.0 (5)_016.fzp
 - Thesis vissim 3.0 (5)_017.fzp
 - Thesis vissim 3.0 (5)_018.fzp
 - Thesis vissim 3.0 (5)_019.fzp
 - Thesis vissim 3.0 (5)_020.fzp
 - Thesis vissim 3.0 (5)_021.fzp
 - Thesis vissim 3.0 (5)_022.fzp
 - Thesis vissim 3.0 (5)_023.fzp
 - Thesis vissim 3.0 (5)_024.fzp
 - Thesis vissim 3.0 (5)_025.fzp
 - Thesis vissim 3.0 (5)_026.fzp
 - Thesis vissim 3.0 (5)_027.fzp
 - Thesis vissim 3.0 (5)_028.fzp
 - 

In [ ]:
import pandas as pd
import glob, re, os

BASE_DIR = "/content/drive/My Drive/Colab Notebooks/Safety Vissim"
OUTPUT_DIR = f"{BASE_DIR}/Outputs"
GEOM_PATH = f"{BASE_DIR}/conflict_area_geometry_with_angle.csv"
FZP_GLOB = f"{BASE_DIR}/*.fzp"

fzp_files = sorted(glob.glob(FZP_GLOB))
geom_df = pd.read_csv(GEOM_PATH)
geom_indexed = geom_df.set_index("ConflictArea")

labeled_dataset = pd.read_csv(f"{OUTPUT_DIR}/traffic_conflict_dataset_final.csv")

def run_id_from_filename(path):
    m = re.search(r"_(\d{3})\.fzp$", path)
    return int(m.group(1)) if m else None

run_files = {run_id_from_filename(f): f for f in fzp_files}

print(f"Reloaded labeled_dataset: {len(labeled_dataset):,} rows")
print(f"Reloaded {len(fzp_files)} fzp file paths, {len(geom_df)} conflict areas")

Reloaded labeled_dataset: 106,095 rows
Reloaded 30 fzp file paths, 103 conflict areas


In [ ]:
def classify_movement(angle_deg):
    if angle_deg < MERGE_ANGLE_MAX:
        return "MERGE"
    if angle_deg < OBLIQUE_ANGLE_MAX:
        return "OBLIQUE_CROSS"
    return "PERPENDICULAR_CROSS"

In [ ]:
import pandas as pd
import numpy as np

CHUNK_SIZE = 500000
LOOKBACK_SECONDS = 8.0
APPROACH_WINDOW = 3.0
STOP_SPEED_KMH = 0.5
MERGE_ANGLE_MAX = 20.0
OBLIQUE_ANGLE_MAX = 60.0
VEHICLE_LENGTH_M = 4.5

FZP_HEADER = ["SIMSEC", "NO", "LINK", "LANE_INDEX", "POS", "POSLAT", "ACCEL", "CFX", "CFY", "CFZ", "CRX", "CRY", "CRZ", "SPEED"]

def load_needed_trajectories(fzp_path, needed_keys):
    header_line_idx = None
    f = open(fzp_path, encoding="utf-8-sig")
    for i, line in enumerate(f):
        if line.startswith("$VEHICLE:"):
            header_line_idx = i
            break
    f.close()
    needed_vehs = set(v for v, l in needed_keys)
    needed_links = set(l for v, l in needed_keys)
    collected = dict((k, []) for k in needed_keys)
    reader = pd.read_csv(fzp_path, sep=";", skiprows=header_line_idx + 1, names=FZP_HEADER, encoding="utf-8-sig", chunksize=CHUNK_SIZE, engine="c", usecols=["SIMSEC", "NO", "LINK", "POS", "SPEED"])
    for chunk in reader:
        chunk = chunk[chunk["NO"].isin(needed_vehs) & chunk["LINK"].isin(needed_links)]

In [ ]:
APPROACH_WINDOW = 3.0
STOP_SPEED_KMH = 0.5
MERGE_ANGLE_MAX = 20.0
OBLIQUE_ANGLE_MAX = 60.0

def compute_decel_rate(traj, cross_time, window):
    seg = traj[(traj["SIMSEC"] >= cross_time - window) & (traj["SIMSEC"] <= cross_time)]
    if len(seg) < 2:
        return np.nan
    seg = seg.sort_values("SIMSEC")
    v0 = seg["SPEED"].iloc[0] / 3.6
    v1 = seg["SPEED"].iloc[-1] / 3.6
    dt = seg["SIMSEC"].iloc[-1] - seg["SIMSEC"].iloc[0]
    if dt <= 0:
        return np.nan
    return round((v0 - v1) / dt, 3)

def compute_wait_time(traj, cross_time, lookback):
    seg = traj[(traj["SIMSEC"] >= cross_time - lookback) & (traj["SIMSEC"] <= cross_time)]
    if seg.empty:
        return 0.0
    seg = seg.sort_values("SIMSEC")
    stopped = seg["SPEED"] < STOP_SPEED_KMH
    if not stopped.any():
        return 0.0
    dt = seg["SIMSEC"].diff().fillna(0)
    return round(float(dt[stopped].sum()), 2)

def classify_movement(angle_deg):
    if angle_deg < MERGE_ANGLE_MAX:
        return "MERGE"
    elif angle_deg < OBLIQUE_ANGLE_MAX:
        return "OBLIQUE_CROSS"
    else:
        return "PERPENDICULAR_CROSS"

featured = labeled_dataset.join(geom_indexed[["CrossingAngleDeg"]], on="ConflictArea")
featured["MovementType"] = featured["CrossingAngleDeg"].apply(classify_movement)

decel_a_map, decel_b_map, wait_a_map, wait_b_map = {}, {}, {}, {}

for run_id, run_rows in featured.groupby("RunID"):
    if run_id not in run_files:
        continue
    fzp_path = run_files[run_id]
    print(f"\n--- RunID {run_id}: {len(run_rows)} rows ---")

    needed_keys = set()
    for _, row in run_rows.iterrows():
        needed_keys.add((row.VehA, row.LinkA))
        needed_keys.add((row.VehB, row.LinkB))

    trajectories = load_needed_trajectories(fzp_path, needed_keys)
    print(f"  Loaded {len(trajectories)} trajectory segments")

    for idx, row in run_rows.iterrows():
        key_a = (row.VehA, row.LinkA)
        key_b = (row.VehB, row.LinkB)
        if key_a in trajectories:
            decel_a_map[idx] = compute_decel_rate(trajectories[key_a], row.CrossTimeA, APPROACH_WINDOW)
            wait_a_map[idx] = compute_wait_time(trajectories[key_a], row.CrossTimeA, LOOKBACK_SECONDS)
        if key_b in trajectories:
            decel_b_map[idx] = compute_decel_rate(trajectories[key_b], row.CrossTimeB, APPROACH_WINDOW)
            wait_b_map[idx] = compute_wait_time(trajectories[key_b], row.CrossTimeB, LOOKBACK_SECONDS)

featured["DecelRate_A"] = pd.Series(decel_a_map)
featured["DecelRate_B"] = pd.Series(decel_b_map)
featured["WaitTime_A"] = pd.Series(wait_a_map).fillna(0.0)
featured["WaitTime_B"] = pd.Series(wait_b_map).fillna(0.0)

print(f"\nMovement type distribution:\n{featured['MovementType'].value_counts()}")
print(f"\nMissing DecelRate: A={featured.DecelRate_A.isna().sum()}, B={featured.DecelRate_B.isna().sum()}")
print(f"\nConflict rate by whether vehicle A queued:")
_queued_check = featured.WaitTime_A > 0
print(featured.groupby(_queued_check)['Conflict'].mean())

featured.to_csv(f"{OUTPUT_DIR}/traffic_conflict_dataset_features.csv", index=False)
print(f"\n*** DONE. Final feature dataset saved to {OUTPUT_DIR}/traffic_conflict_dataset_features.csv ***")
print(f"*** NOTE: TimeGap/AcceptedGap, TTC, and PET are inputs to the Conflict label ***")
print(f"*** and must be EXCLUDED from ML model features to avoid data leakage.        ***")
print(f"*** (See Stage 5 preprocessing script for the correct feature list.)          ***")


--- RunID 1: 3949 rows ---


TypeError: object of type 'NoneType' has no len()

In [ ]:
exec(open("/content/drive/My Drive/Colab Notebooks/Safety Vissim/cell5_helpers.py").read())

All Cell 5 dependencies loaded from file successfully.


In [ ]:
APPROACH_WINDOW = 3.0
STOP_SPEED_KMH = 0.5
MERGE_ANGLE_MAX = 20.0
OBLIQUE_ANGLE_MAX = 60.0

def compute_decel_rate(traj, cross_time, window):
    seg = traj[(traj["SIMSEC"] >= cross_time - window) & (traj["SIMSEC"] <= cross_time)]
    if len(seg) < 2:
        return np.nan
    seg = seg.sort_values("SIMSEC")
    v0 = seg["SPEED"].iloc[0] / 3.6
    v1 = seg["SPEED"].iloc[-1] / 3.6
    dt = seg["SIMSEC"].iloc[-1] - seg["SIMSEC"].iloc[0]
    if dt <= 0:
        return np.nan
    return round((v0 - v1) / dt, 3)

def compute_wait_time(traj, cross_time, lookback):
    seg = traj[(traj["SIMSEC"] >= cross_time - lookback) & (traj["SIMSEC"] <= cross_time)]
    if seg.empty:
        return 0.0
    seg = seg.sort_values("SIMSEC")
    stopped = seg["SPEED"] < STOP_SPEED_KMH
    if not stopped.any():
        return 0.0
    dt = seg["SIMSEC"].diff().fillna(0)
    return round(float(dt[stopped].sum()), 2)

def classify_movement(angle_deg):
    if angle_deg < MERGE_ANGLE_MAX:
        return "MERGE"
    elif angle_deg < OBLIQUE_ANGLE_MAX:
        return "OBLIQUE_CROSS"
    else:
        return "PERPENDICULAR_CROSS"

featured = labeled_dataset.join(geom_indexed[["CrossingAngleDeg"]], on="ConflictArea")
featured["MovementType"] = featured["CrossingAngleDeg"].apply(classify_movement)

decel_a_map, decel_b_map, wait_a_map, wait_b_map = {}, {}, {}, {}

for run_id, run_rows in featured.groupby("RunID"):
    if run_id not in run_files:
        continue
    fzp_path = run_files[run_id]
    print(f"\n--- RunID {run_id}: {len(run_rows)} rows ---")

    needed_keys = set()
    for _, row in run_rows.iterrows():
        needed_keys.add((row.VehA, row.LinkA))
        needed_keys.add((row.VehB, row.LinkB))

    trajectories = load_needed_trajectories(fzp_path, needed_keys)
    print(f"  Loaded {len(trajectories)} trajectory segments")

    for idx, row in run_rows.iterrows():
        key_a = (row.VehA, row.LinkA)
        key_b = (row.VehB, row.LinkB)
        if key_a in trajectories:
            decel_a_map[idx] = compute_decel_rate(trajectories[key_a], row.CrossTimeA, APPROACH_WINDOW)
            wait_a_map[idx] = compute_wait_time(trajectories[key_a], row.CrossTimeA, LOOKBACK_SECONDS)
        if key_b in trajectories:
            decel_b_map[idx] = compute_decel_rate(trajectories[key_b], row.CrossTimeB, APPROACH_WINDOW)
            wait_b_map[idx] = compute_wait_time(trajectories[key_b], row.CrossTimeB, LOOKBACK_SECONDS)

featured["DecelRate_A"] = pd.Series(decel_a_map)
featured["DecelRate_B"] = pd.Series(decel_b_map)
featured["WaitTime_A"] = pd.Series(wait_a_map).fillna(0.0)
featured["WaitTime_B"] = pd.Series(wait_b_map).fillna(0.0)

print(f"\nMovement type distribution:\n{featured['MovementType'].value_counts()}")
print(f"\nMissing DecelRate: A={featured.DecelRate_A.isna().sum()}, B={featured.DecelRate_B.isna().sum()}")
print(f"\nConflict rate by whether vehicle A queued:")
_queued_check = featured.WaitTime_A > 0
print(featured.groupby(_queued_check)['Conflict'].mean())

featured.to_csv(f"{OUTPUT_DIR}/traffic_conflict_dataset_features.csv", index=False)
print(f"\n*** DONE. Final feature dataset saved to {OUTPUT_DIR}/traffic_conflict_dataset_features.csv ***")
print(f"*** NOTE: TimeGap/AcceptedGap, TTC, and PET are inputs to the Conflict label ***")
print(f"*** and must be EXCLUDED from ML model features to avoid data leakage.        ***")
print(f"*** (See Stage 5 preprocessing script for the correct feature list.)          ***")


--- RunID 1: 3949 rows ---
  Loaded 2494 trajectory segments

--- RunID 2: 2906 rows ---
  Loaded 1761 trajectory segments

--- RunID 3: 3449 rows ---
  Loaded 2163 trajectory segments

--- RunID 4: 3442 rows ---
  Loaded 2210 trajectory segments

--- RunID 5: 3858 rows ---
  Loaded 2485 trajectory segments

--- RunID 6: 3179 rows ---
  Loaded 2012 trajectory segments

--- RunID 7: 3559 rows ---
  Loaded 2322 trajectory segments

--- RunID 8: 3494 rows ---
  Loaded 2144 trajectory segments

--- RunID 9: 3016 rows ---
  Loaded 1994 trajectory segments

--- RunID 10: 3565 rows ---
  Loaded 2162 trajectory segments

--- RunID 11: 3647 rows ---
  Loaded 2330 trajectory segments

--- RunID 12: 3757 rows ---
  Loaded 2338 trajectory segments

--- RunID 13: 3876 rows ---
  Loaded 2395 trajectory segments

--- RunID 14: 3754 rows ---
  Loaded 2368 trajectory segments

--- RunID 15: 3549 rows ---
  Loaded 2347 trajectory segments

--- RunID 16: 3391 rows ---
  Loaded 2275 trajectory segments



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, glob
import numpy as np
import pandas as pd

BASE_DIR = "/content/drive/My Drive/Colab Notebooks/Safety Vissim"
OUTPUT_DIR = f"{BASE_DIR}/Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GEOM_PATH = f"{BASE_DIR}/conflict_area_geometry_with_angle.csv"
FZP_GLOB = f"{BASE_DIR}/*.fzp"

# From VISSIM Base Data > Vehicle Types (confirmed against your network)
VEHTYPE_NAMES = {100: "Car", 300: "Bus", 610: "Bike", 620: "CNG", 630: "Pickup", 640: "Rickshaw"}

CHUNK_SIZE = 500_000
FZP_HEADER = ["SIMSEC", "NO", "LINK", "LANE_INDEX", "POS", "POSLAT", "ACCEL",
              "CFX", "CFY", "CFZ", "CRX", "CRY", "CRZ", "SPEED", "VEHTYPE"]

fzp_files = sorted(glob.glob(FZP_GLOB))
print(f"Found {len(fzp_files)} fzp files")
print(f"Geometry file exists: {os.path.exists(GEOM_PATH)}")
for f in fzp_files:
    print(" -", os.path.basename(f))

Mounted at /content/drive
Found 30 fzp files
Geometry file exists: True
 - Thesis vissim 3.0 (5)_001.fzp
 - Thesis vissim 3.0 (5)_002.fzp
 - Thesis vissim 3.0 (5)_003.fzp
 - Thesis vissim 3.0 (5)_004.fzp
 - Thesis vissim 3.0 (5)_005.fzp
 - Thesis vissim 3.0 (5)_006.fzp
 - Thesis vissim 3.0 (5)_007.fzp
 - Thesis vissim 3.0 (5)_008.fzp
 - Thesis vissim 3.0 (5)_009.fzp
 - Thesis vissim 3.0 (5)_010.fzp
 - Thesis vissim 3.0 (5)_011.fzp
 - Thesis vissim 3.0 (5)_012.fzp
 - Thesis vissim 3.0 (5)_013.fzp
 - Thesis vissim 3.0 (5)_014.fzp
 - Thesis vissim 3.0 (5)_015.fzp
 - Thesis vissim 3.0 (5)_016.fzp
 - Thesis vissim 3.0 (5)_017.fzp
 - Thesis vissim 3.0 (5)_018.fzp
 - Thesis vissim 3.0 (5)_019.fzp
 - Thesis vissim 3.0 (5)_020.fzp
 - Thesis vissim 3.0 (5)_021.fzp
 - Thesis vissim 3.0 (5)_022.fzp
 - Thesis vissim 3.0 (5)_023.fzp
 - Thesis vissim 3.0 (5)_024.fzp
 - Thesis vissim 3.0 (5)_025.fzp
 - Thesis vissim 3.0 (5)_026.fzp
 - Thesis vissim 3.0 (5)_027.fzp
 - Thesis vissim 3.0 (5)_028.fzp
 - T

In [3]:
def run_id_from_filename(path):
    m = re.search(r"_(\d{3})\.fzp$", path)
    return int(m.group(1)) if m else None

def build_link_station_lookup(geom_df):
    lookup = {}
    for _, row in geom_df.iterrows():
        lookup.setdefault(int(row.LinkA), []).append((int(row.ConflictArea), row.StationA))
        lookup.setdefault(int(row.LinkB), []).append((int(row.ConflictArea), row.StationB))
    return lookup

def process_one_file(fzp_path, run_id, link_lookup, relevant_links):
    print(f"\n--- Processing RunID {run_id}: {os.path.basename(fzp_path)} ---")
    with open(fzp_path, encoding="utf-8-sig") as f:
        header_line_idx = None
        for i, line in enumerate(f):
            if line.startswith("$VEHICLE:"):
                header_line_idx = i
                break

    crossing_records = []
    last_seen = {}
    veh_type_lookup = {}  # veh NO -> VEHTYPE code (constant per vehicle across its trip)

    reader = pd.read_csv(
        fzp_path, sep=";", skiprows=header_line_idx + 1, names=FZP_HEADER,
        encoding="utf-8-sig", chunksize=CHUNK_SIZE, engine="c",
    )

    total_rows = 0
    for chunk_no, chunk in enumerate(reader):
        total_rows += len(chunk)

        new_vehs = chunk[~chunk["NO"].isin(veh_type_lookup.keys())]
        for veh, vtype in new_vehs[["NO", "VEHTYPE"]].drop_duplicates("NO").values:
            veh_type_lookup[veh] = vtype

        chunk_f = chunk[chunk["LINK"].isin(relevant_links)]
        if chunk_f.empty:
            continue
        chunk_f = chunk_f.sort_values(["NO", "LINK", "SIMSEC"])

        for (veh, link), grp in chunk_f.groupby(["NO", "LINK"], sort=False):
            grp = grp.reset_index(drop=True)
            key = (veh, link)
            prev = last_seen.get(key)
            rows = []
            if prev is not None:
                rows.append(prev)
            rows.extend(grp.to_dict("records"))

            stations = link_lookup[link]
            for i in range(len(rows) - 1):
                r0, r1 = rows[i], rows[i + 1]
                pos0, pos1 = r0["POS"], r1["POS"]
                t0, t1 = r0["SIMSEC"], r1["SIMSEC"]
                sp0, sp1 = r0["SPEED"], r1["SPEED"]
                if pos1 == pos0:
                    continue
                for ca_no, station in stations:
                    if (pos0 - station) * (pos1 - station) <= 0:
                        frac = (station - pos0) / (pos1 - pos0)
                        cross_time = t0 + frac * (t1 - t0)
                        cross_speed = sp0 + frac * (sp1 - sp0)
                        crossing_records.append({
                            "RunID": run_id, "ConflictArea": ca_no, "Link": link,
                            "Veh": veh, "CrossTime": round(cross_time, 3),
                            "CrossSpeed_kmh": round(cross_speed, 2),
                            "VehType": veh_type_lookup.get(veh, np.nan),
                        })
            last_seen[key] = rows[-1]

        if chunk_no % 3 == 0:
            print(f"  ...{total_rows:,} rows scanned so far, {len(crossing_records):,} crossings found")

    df = pd.DataFrame(crossing_records).drop_duplicates(subset=["RunID", "ConflictArea", "Link", "Veh"])
    df["VehTypeName"] = df["VehType"].map(VEHTYPE_NAMES)
    print(f"  Run {run_id} complete: {len(df):,} crossing events")
    print(f"  Vehicle type breakdown: {df['VehTypeName'].value_counts().to_dict()}")
    return df


geom_df = pd.read_csv(GEOM_PATH)
link_lookup = build_link_station_lookup(geom_df)
relevant_links = set(link_lookup.keys())
print(f"{len(relevant_links)} distinct links/connectors involved in conflict areas")

all_crossing_dfs = []
for fzp_path in fzp_files:
    run_id = run_id_from_filename(fzp_path)
    if run_id is None:
        print(f"WARNING: could not parse RunID from {fzp_path}, skipping")
        continue
    df = process_one_file(fzp_path, run_id, link_lookup, relevant_links)
    all_crossing_dfs.append(df)

vehicle_crossings = pd.concat(all_crossing_dfs, ignore_index=True)
vehicle_crossings.to_csv(f"{OUTPUT_DIR}/vehicle_crossing_times_with_type.csv", index=False)

print(f"\n=== Combined across {len(fzp_files)} runs ===")
print(f"Total crossing events: {len(vehicle_crossings):,}")
print(f"\nOverall vehicle type distribution:")
print(vehicle_crossings["VehTypeName"].value_counts())
print(f"\nSaved to {OUTPUT_DIR}/vehicle_crossing_times_with_type.csv")

33 distinct links/connectors involved in conflict areas

--- Processing RunID 1: Thesis vissim 3.0 (5)_001.fzp ---
  ...500,000 rows scanned so far, 7,992 crossings found
  Run 1 complete: 21,428 crossing events
  Vehicle type breakdown: {'CNG': 8585, 'Bike': 5892, 'Rickshaw': 4876, 'Car': 1634, 'Pickup': 279, 'Bus': 162}

--- Processing RunID 2: Thesis vissim 3.0 (5)_002.fzp ---
  ...500,000 rows scanned so far, 11,318 crossings found
  Run 2 complete: 21,116 crossing events
  Vehicle type breakdown: {'CNG': 8114, 'Bike': 5995, 'Rickshaw': 4930, 'Car': 1725, 'Pickup': 214, 'Bus': 138}

--- Processing RunID 3: Thesis vissim 3.0 (5)_003.fzp ---
  ...500,000 rows scanned so far, 11,163 crossings found
  Run 3 complete: 20,029 crossing events
  Vehicle type breakdown: {'CNG': 7637, 'Bike': 5810, 'Rickshaw': 4517, 'Car': 1551, 'Pickup': 331, 'Bus': 183}

--- Processing RunID 4: Thesis vissim 3.0 (5)_004.fzp ---
  ...500,000 rows scanned so far, 10,943 crossings found
  Run 4 complete: 20,1

In [4]:
TTC_PET_THRESHOLD = 1.5  # seconds; SSAM-style conflict cutoff (Gettman & Head, 2003)

ca_links = geom_df.set_index("ConflictArea")[["LinkA", "LinkB"]].to_dict("index")

records = []
skipped_same_link = 0

for (run_id, ca), grp in vehicle_crossings.groupby(["RunID", "ConflictArea"]):
    if ca not in ca_links:
        continue
    link_a_id = ca_links[ca]["LinkA"]
    link_b_id = ca_links[ca]["LinkB"]

    approach_a = grp[grp.Link == link_a_id].sort_values("CrossTime").reset_index(drop=True)
    approach_b = grp[grp.Link == link_b_id].sort_values("CrossTime").reset_index(drop=True)
    if approach_a.empty or approach_b.empty:
        continue

    combined = pd.concat([approach_a.assign(Approach="A"), approach_b.assign(Approach="B")])
    combined = combined.sort_values("CrossTime").reset_index(drop=True)

    n = len(combined)
    for i in range(n - 1):
        veh_a = combined.loc[i]
        veh_b = combined.loc[i + 1]
        if veh_a.Approach == veh_b.Approach:
            skipped_same_link += 1
            continue
        gap = veh_b.CrossTime - veh_a.CrossTime

        speed_a_ms = veh_a.CrossSpeed_kmh / 3.6
        speed_b_ms = veh_b.CrossSpeed_kmh / 3.6
        rel_speed = abs(speed_a_ms - speed_b_ms)

        if gap <= TTC_PET_THRESHOLD:
            gap_distance = gap * speed_a_ms
            ttc = gap_distance / rel_speed if rel_speed > 1e-3 else gap
            pet = np.nan
        else:
            pet = gap
            ttc = np.nan

        conflict = int(
            (not np.isnan(ttc) and ttc < TTC_PET_THRESHOLD) or
            (not np.isnan(pet) and pet < TTC_PET_THRESHOLD)
        )

        records.append({
            "RunID": run_id, "ConflictArea": ca, "LinkA": veh_a.Link, "LinkB": veh_b.Link,
            "VehA": int(veh_a.Veh), "VehB": int(veh_b.Veh),
            "VehTypeA": veh_a.VehTypeName, "VehTypeB": veh_b.VehTypeName,
            "CrossTimeA": veh_a.CrossTime, "CrossTimeB": veh_b.CrossTime,
            "TimeGap": round(gap, 3), "SpeedA_kmh": veh_a.CrossSpeed_kmh, "SpeedB_kmh": veh_b.CrossSpeed_kmh,
            "RelativeSpeed_ms": round(rel_speed, 3),
            "TTC": round(ttc, 3) if not np.isnan(ttc) else np.nan,
            "PET": round(pet, 3) if not np.isnan(pet) else np.nan,
            "Conflict": conflict,
        })

labeled_dataset = pd.DataFrame(records)

print(f"Skipped {skipped_same_link:,} same-approach pairs (car-following)")
print(f"\nTotal vehicle-pair interactions: {len(labeled_dataset):,}")
print(f"\nPer-run breakdown:")
print(labeled_dataset.groupby("RunID").agg(interactions=("Conflict", "size"), conflicts=("Conflict", "sum")))
print(f"\nOverall conflict rate: {labeled_dataset.Conflict.mean():.1%}")

print(f"\nConflict rate by VehTypeA x VehTypeB pairing (top 15 by interaction count):")
pairing_stats = labeled_dataset.groupby(["VehTypeA", "VehTypeB"]).agg(
    n=("Conflict", "size"), conflict_rate=("Conflict", "mean")
).sort_values("n", ascending=False)
print(pairing_stats.head(15).to_string())

labeled_dataset.to_csv(f"{OUTPUT_DIR}/traffic_conflict_dataset_with_vehtype.csv", index=False)
print(f"\nSaved to {OUTPUT_DIR}/traffic_conflict_dataset_with_vehtype.csv")

Skipped 360,184 same-approach pairs (car-following)

Total vehicle-pair interactions: 138,048

Per-run breakdown:
       interactions  conflicts
RunID                         
1              4520        807
2              5183        764
3              4527        786
4              4482        764
5              4578        887
6              4992        843
7              4546        695
8              4434        750
9              4715        835
10             4326        672
11             4422        733
12             4262        763
13             4583        745
14             4609        785
15             4190        719
16             4953        834
17             4391        802
18             4947        778
19             4154        789
20             4501        847
21             4605        729
22             4781        802
23             4442        771
24             4977        949
25             4749        805
26             4617        815
27             458

In [5]:
print(vehicle_crossings["VehTypeName"].isna().sum(), "/", len(vehicle_crossings), "crossings missing vehicle type")
print(vehicle_crossings[vehicle_crossings["VehTypeName"].isna()]["VehType"].unique())

0 / 617749 crossings missing vehicle type
[]


In [6]:
LOOKBACK_SECONDS = 8.0
VEHICLE_LENGTH_M = 4.5

def load_needed_trajectories(fzp_path, needed_keys):
    with open(fzp_path, encoding="utf-8-sig") as f:
        header_line_idx = None
        header_line = None
        for i, line in enumerate(f):
            if line.startswith("$VEHICLE:"):
                header_line_idx = i
                header_line = line
                break
    # detect actual column count in this file's header, in case it differs
    # (e.g. older exports without the VEHTYPE column)
    n_cols = len(header_line.strip().split(";"))
    file_header = FZP_HEADER[:n_cols]

    needed_vehs = {v for v, l in needed_keys}
    needed_links = {l for v, l in needed_keys}
    collected = {k: [] for k in needed_keys}

    reader = pd.read_csv(
        fzp_path, sep=";", skiprows=header_line_idx + 1, names=file_header,
        encoding="utf-8-sig", chunksize=CHUNK_SIZE, engine="c",
        usecols=["SIMSEC", "NO", "LINK", "POS", "SPEED"],
    )
    for chunk in reader:
        chunk = chunk[chunk["NO"].isin(needed_vehs) & chunk["LINK"].isin(needed_links)]
        if chunk.empty:
            continue
        for (veh, link), grp in chunk.groupby(["NO", "LINK"]):
            key = (veh, link)
            if key in collected:
                collected[key].append(grp[["SIMSEC", "POS", "SPEED"]])

    result = {}
    for key, parts in collected.items():
        if parts:
            df = pd.concat(parts, ignore_index=True).sort_values("SIMSEC").drop_duplicates("SIMSEC")
            result[key] = df.reset_index(drop=True)
    return result

def compute_crossing_ttc(traj_1, station_1, traj_2, station_2, lookback_seconds):
    df1 = traj_1.copy(); df1["dist_to_conflict"] = station_1 - df1["POS"]
    df2 = traj_2.copy(); df2["dist_to_conflict"] = station_2 - df2["POS"]

    t_cross = max(df1["SIMSEC"].max(), df2["SIMSEC"].max())
    t_min = t_cross - lookback_seconds

    df1 = df1[(df1["SIMSEC"] >= t_min) & (df1["dist_to_conflict"] >= 0) & (df1["SPEED"] > 0.5)]
    df2 = df2[(df2["SIMSEC"] >= t_min) & (df2["dist_to_conflict"] >= 0) & (df2["SPEED"] > 0.5)]
    if df1.empty or df2.empty:
        return np.nan

    merged = pd.merge(df1, df2, on="SIMSEC", suffixes=("_1", "_2"))
    if merged.empty:
        return np.nan

    speed_1_ms = merged["SPEED_1"] / 3.6
    speed_2_ms = merged["SPEED_2"] / 3.6
    entry_time_1 = merged["dist_to_conflict_1"] / speed_1_ms
    entry_time_2 = merged["dist_to_conflict_2"] / speed_2_ms
    exit_time_1 = entry_time_1 + (VEHICLE_LENGTH_M / speed_1_ms)
    exit_time_2 = entry_time_2 + (VEHICLE_LENGTH_M / speed_2_ms)

    veh1_first = entry_time_1 < entry_time_2
    ttc_values = pd.Series(np.nan, index=merged.index)

    case_a = veh1_first & (exit_time_1 > entry_time_2)
    ttc_values[case_a] = entry_time_2[case_a]
    case_b = (~veh1_first) & (exit_time_2 > entry_time_1)
    ttc_values[case_b] = entry_time_1[case_b]

    ttc_values = ttc_values.dropna()
    return float(ttc_values.min()) if not ttc_values.empty else np.nan


geom_indexed = geom_df.set_index("ConflictArea")
candidates = labeled_dataset[labeled_dataset["TTC"].notna()].copy()
print(f"{len(candidates):,} candidate pairs need trajectory-based TTC")

labeled_dataset["TTC_method"] = np.where(labeled_dataset["TTC"].notna(), "gap_approx", "not_applicable")

run_files = {run_id_from_filename(f): f for f in fzp_files}
results = {}
not_defined_count = 0

for run_id, run_candidates in candidates.groupby("RunID"):
    if run_id not in run_files:
        continue
    fzp_path = run_files[run_id]
    print(f"\n--- RunID {run_id}: {len(run_candidates)} candidate pairs ---")

    needed_keys = set()
    for _, row in run_candidates.iterrows():
        needed_keys.add((row.VehA, row.LinkA))
        needed_keys.add((row.VehB, row.LinkB))

    trajectories = load_needed_trajectories(fzp_path, needed_keys)
    print(f"  Loaded {len(trajectories)} trajectory segments")

    for idx, row in run_candidates.iterrows():
        ca = row.ConflictArea
        if ca not in geom_indexed.index:
            continue
        station_a = geom_indexed.loc[ca, "StationA"] if geom_indexed.loc[ca, "LinkA"] == row.LinkA else geom_indexed.loc[ca, "StationB"]
        station_b = geom_indexed.loc[ca, "StationB"] if geom_indexed.loc[ca, "LinkB"] == row.LinkB else geom_indexed.loc[ca, "StationA"]

        key_a = (row.VehA, row.LinkA)
        key_b = (row.VehB, row.LinkB)
        if key_a not in trajectories or key_b not in trajectories:
            continue

        ttc_val = compute_crossing_ttc(trajectories[key_a], station_a, trajectories[key_b], station_b, LOOKBACK_SECONDS)
        if not np.isnan(ttc_val):
            results[idx] = ttc_val
        else:
            not_defined_count += 1

print(f"\nTrajectory-based TTC defined for {len(results):,} / {len(candidates):,} candidates")

for idx, val in results.items():
    labeled_dataset.loc[idx, "TTC"] = round(val, 3)
    labeled_dataset.loc[idx, "TTC_method"] = "trajectory"

labeled_dataset["Conflict"] = (
    ((labeled_dataset["TTC"].notna()) & (labeled_dataset["TTC"] < 1.5)) |
    ((labeled_dataset["PET"].notna()) & (labeled_dataset["PET"] < 1.5))
).astype(int)

print(f"\nTTC method breakdown:\n{labeled_dataset['TTC_method'].value_counts()}")
print(f"\nUpdated conflict rate: {labeled_dataset['Conflict'].mean():.1%}")

labeled_dataset.to_csv(f"{OUTPUT_DIR}/traffic_conflict_dataset_final_with_vehtype.csv", index=False)
print(f"\nSaved to {OUTPUT_DIR}/traffic_conflict_dataset_final_with_vehtype.csv")

77,734 candidate pairs need trajectory-based TTC

--- RunID 1: 2600 candidate pairs ---
  Loaded 2354 trajectory segments

--- RunID 2: 2738 candidate pairs ---
  Loaded 2424 trajectory segments

--- RunID 3: 2455 candidate pairs ---
  Loaded 2227 trajectory segments

--- RunID 4: 2513 candidate pairs ---
  Loaded 2305 trajectory segments

--- RunID 5: 2751 candidate pairs ---
  Loaded 2476 trajectory segments

--- RunID 6: 2760 candidate pairs ---
  Loaded 2484 trajectory segments

--- RunID 7: 2441 candidate pairs ---
  Loaded 2132 trajectory segments

--- RunID 8: 2496 candidate pairs ---
  Loaded 2251 trajectory segments

--- RunID 9: 2678 candidate pairs ---
  Loaded 2305 trajectory segments

--- RunID 10: 2273 candidate pairs ---
  Loaded 2126 trajectory segments

--- RunID 11: 2499 candidate pairs ---
  Loaded 2221 trajectory segments

--- RunID 12: 2431 candidate pairs ---
  Loaded 2271 trajectory segments

--- RunID 13: 2492 candidate pairs ---
  Loaded 2243 trajectory segment

In [7]:
APPROACH_WINDOW = 3.0
STOP_SPEED_KMH = 0.5
MERGE_ANGLE_MAX = 20.0
OBLIQUE_ANGLE_MAX = 60.0

def compute_decel_rate(traj, cross_time, window):
    seg = traj[(traj["SIMSEC"] >= cross_time - window) & (traj["SIMSEC"] <= cross_time)]
    if len(seg) < 2:
        return np.nan
    seg = seg.sort_values("SIMSEC")
    v0 = seg["SPEED"].iloc[0] / 3.6
    v1 = seg["SPEED"].iloc[-1] / 3.6
    dt = seg["SIMSEC"].iloc[-1] - seg["SIMSEC"].iloc[0]
    if dt <= 0:
        return np.nan
    return round((v0 - v1) / dt, 3)

def compute_wait_time(traj, cross_time, lookback):
    seg = traj[(traj["SIMSEC"] >= cross_time - lookback) & (traj["SIMSEC"] <= cross_time)]
    if seg.empty:
        return 0.0
    seg = seg.sort_values("SIMSEC")
    stopped = seg["SPEED"] < STOP_SPEED_KMH
    if not stopped.any():
        return 0.0
    dt = seg["SIMSEC"].diff().fillna(0)
    return round(float(dt[stopped].sum()), 2)

def classify_movement(angle_deg):
    if angle_deg < MERGE_ANGLE_MAX:
        return "MERGE"
    if angle_deg < OBLIQUE_ANGLE_MAX:
        return "OBLIQUE_CROSS"
    return "PERPENDICULAR_CROSS"

featured = labeled_dataset.join(geom_indexed[["CrossingAngleDeg"]], on="ConflictArea")
featured["MovementType"] = featured["CrossingAngleDeg"].apply(classify_movement)

# vehicle heterogeneity features
featured["SameVehType"] = (featured["VehTypeA"] == featured["VehTypeB"]).astype(int)
featured["VehTypePair"] = featured["VehTypeA"] + "_" + featured["VehTypeB"]

decel_a_map, decel_b_map, wait_a_map, wait_b_map = {}, {}, {}, {}

for run_id, run_rows in featured.groupby("RunID"):
    if run_id not in run_files:
        continue
    fzp_path = run_files[run_id]
    print(f"\n--- RunID {run_id}: {len(run_rows)} rows ---")

    needed_keys = set()
    for _, row in run_rows.iterrows():
        needed_keys.add((row.VehA, row.LinkA))
        needed_keys.add((row.VehB, row.LinkB))

    trajectories = load_needed_trajectories(fzp_path, needed_keys)
    print(f"  Loaded {len(trajectories)} trajectory segments")

    for idx, row in run_rows.iterrows():
        key_a = (row.VehA, row.LinkA)
        key_b = (row.VehB, row.LinkB)
        if key_a in trajectories:
            decel_a_map[idx] = compute_decel_rate(trajectories[key_a], row.CrossTimeA, APPROACH_WINDOW)
            wait_a_map[idx] = compute_wait_time(trajectories[key_a], row.CrossTimeA, LOOKBACK_SECONDS)
        if key_b in trajectories:
            decel_b_map[idx] = compute_decel_rate(trajectories[key_b], row.CrossTimeB, APPROACH_WINDOW)
            wait_b_map[idx] = compute_wait_time(trajectories[key_b], row.CrossTimeB, LOOKBACK_SECONDS)

featured["DecelRate_A"] = pd.Series(decel_a_map)
featured["DecelRate_B"] = pd.Series(decel_b_map)
featured["WaitTime_A"] = pd.Series(wait_a_map).fillna(0.0)
featured["WaitTime_B"] = pd.Series(wait_b_map).fillna(0.0)

print(f"\nMovement type distribution:\n{featured['MovementType'].value_counts()}")
print(f"\nMissing DecelRate: A={featured.DecelRate_A.isna().sum()}, B={featured.DecelRate_B.isna().sum()}")

print(f"\n=== VEHICLE HETEROGENEITY RESULTS ===")
print(f"\nConflict rate: same vehicle type vs different vehicle type pairing:")
print(featured.groupby("SameVehType")["Conflict"].agg(["mean", "count"]))

print(f"\nConflict rate by individual vehicle type (as VehTypeA, i.e. first-arriving vehicle):")
print(featured.groupby("VehTypeA")["Conflict"].agg(["mean", "count"]).sort_values("mean", ascending=False))

print(f"\nTop 15 vehicle type pairings by conflict rate (min 100 interactions):")
pair_stats = featured.groupby("VehTypePair")["Conflict"].agg(["mean", "count"])
pair_stats = pair_stats[pair_stats["count"] >= 100].sort_values("mean", ascending=False)
print(pair_stats.head(15).to_string())

featured.to_csv(f"{OUTPUT_DIR}/traffic_conflict_dataset_features_with_vehtype.csv", index=False)
print(f"\n*** DONE. Final feature dataset with vehicle heterogeneity saved to: ***")
print(f"*** {OUTPUT_DIR}/traffic_conflict_dataset_features_with_vehtype.csv ***")
print(f"*** NOTE: TimeGap, TTC, and PET must still be EXCLUDED from ML features (data leakage). ***")


--- RunID 1: 4520 rows ---
  Loaded 2716 trajectory segments

--- RunID 2: 5183 rows ---
  Loaded 2802 trajectory segments

--- RunID 3: 4527 rows ---
  Loaded 2610 trajectory segments

--- RunID 4: 4482 rows ---
  Loaded 2680 trajectory segments

--- RunID 5: 4578 rows ---
  Loaded 2820 trajectory segments

--- RunID 6: 4992 rows ---
  Loaded 2858 trajectory segments

--- RunID 7: 4546 rows ---
  Loaded 2467 trajectory segments

--- RunID 8: 4434 rows ---
  Loaded 2599 trajectory segments

--- RunID 9: 4715 rows ---
  Loaded 2688 trajectory segments

--- RunID 10: 4326 rows ---
  Loaded 2522 trajectory segments

--- RunID 11: 4422 rows ---
  Loaded 2567 trajectory segments

--- RunID 12: 4262 rows ---
  Loaded 2563 trajectory segments

--- RunID 13: 4583 rows ---
  Loaded 2627 trajectory segments

--- RunID 14: 4609 rows ---
  Loaded 2722 trajectory segments

--- RunID 15: 4190 rows ---
  Loaded 2492 trajectory segments

--- RunID 16: 4953 rows ---
  Loaded 2796 trajectory segments

